# logs_local_berk: FiLM Runs

Compare FiLM runs (Rosie vs ImageNet weights, z4 vs z1z4 vs z3z4, Baseline).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis")
LOGS_LOCAL_BERK = PROJECT_ROOT / "AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk"

sys.path.insert(0, str(PROJECT_ROOT / "Datasets/pannuke_hf_cellvit/notebooks"))
from notebook_utils import load_runs_dataframe

In [ ]:
runs_df = load_runs_dataframe(LOGS_LOCAL_BERK)
film_methods = ["FiLM", "ProxyFiLM", "LoRA+FiLM", "Baseline"]
df = runs_df[runs_df["method"].isin(film_methods)].copy()
print(f"FiLM/Baseline runs: {len(df)}")

## FiLM Leaderboard (by mPQ)

In [ ]:
lb = df.sort_values("mPQ", ascending=False)
cols = ["run_name", "method", "condition_source", "film_target", "mPQ", "bPQ", "DQ", "SQ"]
lb[[c for c in cols if c in lb.columns]].head(25)

## Baseline vs FiLM (by condition_source)

In [ ]:
base = df[df["method"] == "Baseline"][["run_name", "condition_source", "mPQ", "bPQ"]]
film = df[df["method"].isin(["FiLM", "ProxyFiLM", "LoRA+FiLM"])][["run_name", "condition_source", "film_target", "mPQ", "bPQ"]]
print("Baseline:")
print(base.to_string())
print("\nFiLM (top 15):")
print(film.sort_values("mPQ", ascending=False).head(15).to_string())

## Best per group (condition_source × film_target)

In [ ]:
grp = ["condition_source", "film_target"]
grp = [c for c in grp if c in df.columns]
best_per = df.loc[df.groupby(grp)["mPQ"].idxmax()]
best_per = best_per.sort_values("mPQ", ascending=False)
best_per[["run_name", "method", "condition_source", "film_target", "mPQ"]].head(15)

## Rosie vs ImageNet FiLM (apples-to-apples)

In [ ]:
rosie = df[(df["condition_source"] == "Rosie") & (df["method"] != "Baseline")]
imagenet = df[(df["condition_source"] == "ImageNet") & (df["method"] != "Baseline")]
print("Best Rosie FiLM:", rosie.loc[rosie["mPQ"].idxmax(), "run_name"], "mPQ=", round(rosie["mPQ"].max(), 4))
print("Best ImageNet FiLM:", imagenet.loc[imagenet["mPQ"].idxmax(), "run_name"], "mPQ=", round(imagenet["mPQ"].max(), 4))